# Enhanced Model Comparison & Analysis

Comprehensive comparison: V1/V4 Linear Regression vs Advanced Gradient Boosting Models
Memory-optimized for VS Code - Incremental processing with extensive visualizations

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import joblib
import pickle
import json
import os
import gc
import psutil
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.max_open_warning'] = 3
plt.rcParams['font.size'] = 10

def get_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1e9

def cleanup():
    plt.close('all')
    gc.collect()
    return get_memory_usage()

print(f"Initial memory: {get_memory_usage():.2f} GB")
print("Memory-optimized notebook initialized")

Initial memory: 0.22 GB
Memory-optimized notebook initialized


## 1. Load Baseline Performance Metrics Only

In [2]:
baseline_v4 = {
    '1h': {'mae': 2.08, 'r2': 0.845, 'name': 'LinearReg-V4'},
    '2h': {'mae': 2.55, 'r2': 0.765, 'name': 'LinearReg-V4'},
    '3h': {'mae': 2.90, 'r2': 0.697, 'name': 'LinearReg-V4'},
    '4h': {'mae': 3.17, 'r2': 0.641, 'name': 'LinearReg-V4'},
    '5h': {'mae': 3.39, 'r2': 0.592, 'name': 'LinearReg-V4'},
    '6h': {'mae': 3.57, 'r2': 0.550, 'name': 'LinearReg-V4'},
    '12h': {'mae': 4.24, 'r2': 0.379, 'name': 'LinearReg-V4'},
    '24h': {'mae': 4.84, 'r2': 0.201, 'name': 'LinearReg-V4'}
}

baseline_v1 = {
    '1h': {'mae': 2.35, 'r2': 0.769, 'name': 'LinearReg-V1'},
    '2h': {'mae': 3.11, 'r2': 0.608, 'name': 'LinearReg-V1'},
    '3h': {'mae': 3.62, 'r2': 0.479, 'name': 'LinearReg-V1'},
    '4h': {'mae': 4.02, 'r2': 0.382, 'name': 'LinearReg-V1'},
    '5h': {'mae': 4.30, 'r2': 0.310, 'name': 'LinearReg-V1'},
    '6h': {'mae': 4.55, 'r2': 0.252, 'name': 'LinearReg-V1'},
    '12h': {'mae': 5.12, 'r2': 0.097, 'name': 'LinearReg-V1'},
    '24h': {'mae': 5.85, 'r2': -0.061, 'name': 'LinearReg-V1'}
}

horizons = ['1h', '2h', '3h', '4h', '5h', '6h', '12h', '24h']
print(f"Loaded baseline metrics for {len(horizons)} horizons")
print(f"V4 baseline: 485 hexagons, 2023-2025 data")
print(f"V1 baseline: Tokyo subset, 2023 data only")
print(f"Memory: {get_memory_usage():.2f} GB")

Loaded baseline metrics for 8 horizons
V4 baseline: 485 hexagons, 2023-2025 data
V1 baseline: Tokyo subset, 2023 data only
Memory: 0.22 GB


## 2. Load Test Labels Once

In [ ]:
print("Loading test labels...")
try:
    with open('../01_ensemble_training/trained/y_data.pkl', 'rb') as f:
        y_data = pickle.load(f)
        y_test = y_data['y_test']
        test_size = len(y_test['target_1h'])
        del y_data
        gc.collect()
    print(f"✓ Loaded test labels: {test_size:,} samples")
except Exception as e:
    print(f"✗ Error loading test labels: {e}")
    print("Please run the training notebook first to generate y_data.pkl")

print(f"Memory: {get_memory_usage():.2f} GB")

## 3. Process Predictions One Horizon at a Time

In [ ]:
results_summary = {}

def process_single_horizon(horizon):
    """Process a single horizon and return metrics"""
    try:
        pred_file = f'../01_ensemble_training/trained/predictions_{horizon}.pkl'
        if not os.path.exists(pred_file):
            print(f"✗ {horizon}: File not found")
            return None
        
        preds = joblib.load(pred_file)
        test_preds = preds['test']
        
        y_true = y_test[f'target_{horizon[:-1]}h']
        
        ensemble_pred = (test_preds['lgb'] + test_preds['xgb'] + test_preds['cat']) / 3
        
        metrics = {
            'lgb': {
                'mae': mean_absolute_error(y_true, test_preds['lgb']),
                'r2': r2_score(y_true, test_preds['lgb'])
            },
            'xgb': {
                'mae': mean_absolute_error(y_true, test_preds['xgb']),
                'r2': r2_score(y_true, test_preds['xgb'])
            },
            'cat': {
                'mae': mean_absolute_error(y_true, test_preds['cat']),
                'r2': r2_score(y_true, test_preds['cat'])
            },
            'ensemble': {
                'mae': mean_absolute_error(y_true, ensemble_pred),
                'r2': r2_score(y_true, ensemble_pred)
            },
            'baseline_v4': baseline_v4[horizon],
            'baseline_v1': baseline_v1[horizon]
        }
        
        del preds, test_preds, ensemble_pred
        gc.collect()
        
        return metrics
        
    except Exception as e:
        print(f"✗ {horizon}: Error - {str(e)[:50]}")
        return None

print("Processing predictions horizon by horizon...")
for horizon in horizons:
    metrics = process_single_horizon(horizon)
    if metrics:
        results_summary[horizon] = metrics
        ens_mae = metrics['ensemble']['mae']
        v4_mae = metrics['baseline_v4']['mae']
        improvement = (v4_mae - ens_mae) / v4_mae * 100
        print(f"✓ {horizon}: Ensemble MAE={ens_mae:.3f} vs V4={v4_mae:.3f} ({improvement:+.1f}%)")
    cleanup()

print(f"\nProcessed {len(results_summary)} horizons")
print(f"Memory: {get_memory_usage():.2f} GB")

## 4. Create Lightweight Comparison Table

In [5]:
if results_summary:
    models = ['lgb', 'xgb', 'cat', 'ensemble', 'baseline_v4', 'baseline_v1']
    mae_data = []
    r2_data = []
    
    for model in models:
        mae_row = [model]
        r2_row = [model]
        for horizon in horizons:
            if horizon in results_summary:
                mae_row.append(results_summary[horizon][model]['mae'])
                r2_row.append(results_summary[horizon][model]['r2'])
            else:
                mae_row.append(np.nan)
                r2_row.append(np.nan)
        mae_data.append(mae_row)
        r2_data.append(r2_row)
    
    mae_df = pd.DataFrame(mae_data, columns=['Model'] + horizons)
    mae_df.set_index('Model', inplace=True)
    
    r2_df = pd.DataFrame(r2_data, columns=['Model'] + horizons)
    r2_df.set_index('Model', inplace=True)
    
    print("\nMAE Comparison (μg/m³):")
    print("="*60)
    print(mae_df.round(3))
    
    print("\nR² Score Comparison:")
    print("="*60)
    print(r2_df.round(3))
    
    mae_df.to_csv('mae_comparison_optimized.csv')
    r2_df.to_csv('r2_comparison_optimized.csv')
    print("\n✓ Saved comparison tables to CSV")

print(f"Memory: {get_memory_usage():.2f} GB")

Memory: 0.22 GB


## 5. Performance Trend Visualization (Lightweight)

In [6]:
if results_summary:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    horizon_nums = [int(h[:-1]) for h in horizons]
    
    all_models = ['baseline_v1', 'baseline_v4', 'lgb', 'xgb', 'cat', 'ensemble']
    model_labels = {
        'baseline_v1': 'V1-LinearReg',
        'baseline_v4': 'V4-LinearReg', 
        'lgb': 'LightGBM',
        'xgb': 'XGBoost',
        'cat': 'CatBoost',
        'ensemble': 'Ensemble'
    }
    
    colors = ['#e377c2', '#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    
    for i, model in enumerate(all_models):
        mae_values = [results_summary[h][model]['mae'] for h in horizons if h in results_summary]
        r2_values = [results_summary[h][model]['r2'] for h in horizons if h in results_summary]
        
        ax1.plot(horizon_nums, mae_values, 'o-', label=model_labels[model], 
                linewidth=2, markersize=6, color=colors[i])
        ax2.plot(horizon_nums, r2_values, 's-', label=model_labels[model], 
                linewidth=2, markersize=6, color=colors[i])
    
    ax1.set_xlabel('Forecast Horizon (hours)', fontsize=12)
    ax1.set_ylabel('MAE (μg/m³)', fontsize=12)
    ax1.set_title('Forecast Degradation - MAE', fontsize=14)
    ax1.legend(loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(horizon_nums)
    
    ax2.set_xlabel('Forecast Horizon (hours)', fontsize=12)
    ax2.set_ylabel('R² Score', fontsize=12)
    ax2.set_title('Forecast Degradation - R² Score', fontsize=14)
    ax2.legend(loc='upper right', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(horizon_nums)
    ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    
    plt.suptitle('All Models Performance Degradation Across Horizons', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig('forecast_degradation_all_models.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Saved forecast degradation analysis")
    
    cleanup()

print(f"Memory: {get_memory_usage():.2f} GB")

Memory: 0.22 GB


## 5a. Forecast Degradation Analysis - All Models

In [ ]:
def calculate_model_correlations(horizon='6h'):
    """Calculate correlations between model predictions"""
    try:
        pred_file = f'../01_ensemble_training/trained/predictions_{horizon}.pkl'
        if not os.path.exists(pred_file):
            return None
        
        preds = joblib.load(pred_file)
        test_preds = preds['test']
        
        ensemble_pred = (test_preds['lgb'] + test_preds['xgb'] + test_preds['cat']) / 3
        
        corr_matrix = np.corrcoef([
            test_preds['lgb'][:1000],
            test_preds['xgb'][:1000],
            test_preds['cat'][:1000],
            ensemble_pred[:1000]
        ])
        
        del preds, test_preds, ensemble_pred
        gc.collect()
        
        return corr_matrix
    except Exception as e:
        print(f"Error calculating correlations: {e}")
        return None

selected_horizons = ['1h', '6h', '12h', '24h']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, horizon in enumerate(selected_horizons):
    corr_matrix = calculate_model_correlations(horizon)
    if corr_matrix is not None:
        ax = axes[idx]
        
        sns.heatmap(corr_matrix, annot=True, fmt='.3f', 
                   xticklabels=['LGB', 'XGB', 'Cat', 'Ens'],
                   yticklabels=['LGB', 'XGB', 'Cat', 'Ens'],
                   cmap='coolwarm', center=0.5, vmin=0, vmax=1,
                   cbar_kws={'label': 'Correlation'},
                   ax=ax)
        
        ax.set_title(f'{horizon} Model Agreement', fontsize=11)
    
    cleanup()

plt.suptitle('Model Prediction Correlations Across Horizons', fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig('model_agreement_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved model agreement analysis")

print(f"Memory: {get_memory_usage():.2f} GB")

## 5b. Model Agreement and Correlation Analysis

In [ ]:
def get_errors_for_horizon(horizon='6h', sample_size=1000):
    """Get sampled errors for all models at a specific horizon"""
    try:
        pred_file = f'../01_ensemble_training/trained/predictions_{horizon}.pkl'
        if not os.path.exists(pred_file):
            return None
        
        preds = joblib.load(pred_file)
        test_preds = preds['test']
        y_true = y_test[f'target_{horizon[:-1]}h']
        
        ensemble_pred = (test_preds['lgb'] + test_preds['xgb'] + test_preds['cat']) / 3
        
        sample_idx = np.random.RandomState(42).choice(len(y_true), 
                                                      min(sample_size, len(y_true)), 
                                                      replace=False)
        
        errors = {
            'LightGBM': (test_preds['lgb'][sample_idx] - y_true[sample_idx]),
            'XGBoost': (test_preds['xgb'][sample_idx] - y_true[sample_idx]),
            'CatBoost': (test_preds['cat'][sample_idx] - y_true[sample_idx]),
            'Ensemble': (ensemble_pred[sample_idx] - y_true[sample_idx])
        }
        
        del preds, test_preds, ensemble_pred
        gc.collect()
        
        return errors
    except Exception as e:
        print(f"Error getting errors: {e}")
        return None

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.ravel()

for idx, horizon in enumerate(horizons):
    errors = get_errors_for_horizon(horizon, sample_size=500)
    if errors:
        ax = axes[idx]
        
        error_data = []
        labels = []
        for model, err in errors.items():
            error_data.append(err)
            labels.append(model[:3])
        
        parts = ax.violinplot(error_data, showmeans=True, showmedians=True)
        
        for pc in parts['bodies']:
            pc.set_facecolor('#9467bd')
            pc.set_alpha(0.6)
        
        ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels)
        ax.set_title(f'{horizon} Error Distribution', fontsize=10)
        ax.set_ylabel('Error (μg/m³)')
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim(-15, 15)
    
    cleanup()

plt.suptitle('Error Distribution Analysis - All Models Across Horizons', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('error_distribution_violin_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved error distribution violin plots")

print(f"Memory: {get_memory_usage():.2f} GB")

## 5c. Error Distribution Analysis - Violin Plots

In [9]:
if results_summary:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, horizon in enumerate(['6h', '24h']):
        ax = axes[idx]
        
        if horizon in results_summary:
            v1_mae = results_summary[horizon]['baseline_v1']['mae']
            v4_mae = results_summary[horizon]['baseline_v4']['mae']
            lgb_mae = results_summary[horizon]['lgb']['mae']
            xgb_mae = results_summary[horizon]['xgb']['mae']
            cat_mae = results_summary[horizon]['cat']['mae']
            ens_mae = results_summary[horizon]['ensemble']['mae']
            
            categories = ['V1\nBaseline', 'V4\nBaseline', 'LightGBM', 'XGBoost', 'CatBoost', 'Ensemble']
            values = [v1_mae, v4_mae, lgb_mae, xgb_mae, cat_mae, ens_mae]
            
            improvements = [0]
            for i in range(1, len(values)):
                improvements.append(values[0] - values[i])
            
            x_pos = np.arange(len(categories))
            colors = ['red'] + ['green' if imp > 0 else 'red' for imp in improvements[1:]]
            
            bars = ax.bar(x_pos, values, color=colors, alpha=0.7)
            
            for i, (bar, val, imp) in enumerate(zip(bars, values, improvements)):
                if i > 0:
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                           f'{val:.2f}\n({imp:+.2f})', ha='center', va='bottom', fontsize=9)
                else:
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                           f'{val:.2f}', ha='center', va='bottom', fontsize=9)
            
            ax.set_xticks(x_pos)
            ax.set_xticklabels(categories)
            ax.set_ylabel('MAE (μg/m³)', fontsize=11)
            ax.set_title(f'{horizon} Improvement Progression', fontsize=12)
            ax.grid(True, alpha=0.3, axis='y')
            
            best_mae = min(values)
            ax.axhline(y=best_mae, color='green', linestyle='--', alpha=0.5, label=f'Best: {best_mae:.2f}')
            ax.axhline(y=v1_mae, color='red', linestyle='--', alpha=0.3, label=f'V1 Baseline: {v1_mae:.2f}')
            ax.legend(loc='upper right', fontsize=9)
    
    plt.suptitle('Model Improvement Waterfall - From V1 Baseline to Ensemble', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('improvement_waterfall.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Saved improvement waterfall chart")
    
    cleanup()

print(f"Memory: {get_memory_usage():.2f} GB")

Memory: 0.24 GB


## 5d. Improvement Waterfall Chart

In [ ]:
def calculate_error_percentiles(horizon='6h'):
    """Calculate error percentiles for all models"""
    try:
        pred_file = f'../01_ensemble_training/trained/predictions_{horizon}.pkl'
        if not os.path.exists(pred_file):
            return None, None
        
        preds = joblib.load(pred_file)
        test_preds = preds['test']
        y_true = y_test[f'target_{horizon[:-1]}h']
        
        ensemble_pred = (test_preds['lgb'] + test_preds['xgb'] + test_preds['cat']) / 3
        
        percentiles = [25, 50, 75, 90, 95, 99]
        results = {}
        
        for model_name, predictions in [('LightGBM', test_preds['lgb']),
                                       ('XGBoost', test_preds['xgb']),
                                       ('CatBoost', test_preds['cat']),
                                       ('Ensemble', ensemble_pred)]:
            abs_errors = np.abs(predictions - y_true)
            results[model_name] = [np.percentile(abs_errors, p) for p in percentiles]
        
        del preds, test_preds, ensemble_pred
        gc.collect()
        
        return results, percentiles
    except Exception as e:
        print(f"Error calculating percentiles: {e}")
        return None, None

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, horizon in enumerate(['1h', '6h', '24h']):
    results, percentiles = calculate_error_percentiles(horizon)
    
    if results:
        ax = axes[idx]
        
        x = np.arange(len(percentiles))
        width = 0.2
        
        for i, (model, values) in enumerate(results.items()):
            ax.bar(x + i * width, values, width, label=model, alpha=0.8)
        
        ax.set_xlabel('Percentile', fontsize=11)
        ax.set_ylabel('Absolute Error (μg/m³)', fontsize=11)
        ax.set_title(f'{horizon} Error Percentiles', fontsize=12)
        ax.set_xticks(x + width * 1.5)
        ax.set_xticklabels([f'{p}th' for p in percentiles])
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')
    
    cleanup()

plt.suptitle('Error Percentile Analysis - Understanding Tail Behavior', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('error_percentile_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved error percentile analysis")

print(f"Memory: {get_memory_usage():.2f} GB")

## 5e. Percentile Error Analysis

In [ ]:
if results_summary:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    horizon_nums = [int(h[:-1]) for h in horizons]
    
    for model in ['ensemble', 'baseline_v4', 'baseline_v1']:
        mae_values = [results_summary[h][model]['mae'] for h in horizons if h in results_summary]
        r2_values = [results_summary[h][model]['r2'] for h in horizons if h in results_summary]
        
        label = model.replace('baseline_', '').replace('ensemble', 'Ensemble')
        ax1.plot(horizon_nums, mae_values, 'o-', label=label, linewidth=2, markersize=6)
        ax2.plot(horizon_nums, r2_values, 's-', label=label, linewidth=2, markersize=6)
    
    ax1.set_xlabel('Forecast Horizon (hours)')
    ax1.set_ylabel('MAE (μg/m³)')
    ax1.set_title('MAE Performance Across Horizons')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(horizon_nums)
    
    ax2.set_xlabel('Forecast Horizon (hours)')
    ax2.set_ylabel('R² Score')
    ax2.set_title('R² Score Across Horizons')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(horizon_nums)
    ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    
    plt.suptitle('Model Performance Comparison: Ensemble vs Linear Regression Baselines', fontsize=14)
    plt.tight_layout()
    plt.savefig('performance_trends_lightweight.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Saved performance trends plot")
    
    cleanup()

print(f"Memory: {get_memory_usage():.2f} GB")

✓ Saved performance trends plot
Memory: 0.65 GB


## 6. Improvement Heatmap

In [ ]:
if results_summary:
    models_to_compare = ['lgb', 'xgb', 'cat', 'ensemble']
    improvement_data = []
    
    for model in models_to_compare:
        mae_improvements = []
        r2_improvements = []
        for horizon in horizons:
            if horizon in results_summary:
                model_mae = results_summary[horizon][model]['mae']
                v4_mae = results_summary[horizon]['baseline_v4']['mae']
                mae_imp = (v4_mae - model_mae) / v4_mae * 100
                mae_improvements.append(mae_imp)
                
                model_r2 = results_summary[horizon][model]['r2']
                v4_r2 = results_summary[horizon]['baseline_v4']['r2']
                r2_imp = (model_r2 - v4_r2) / abs(v4_r2) * 100 if v4_r2 != 0 else 0
                r2_improvements.append(r2_imp)
        
        improvement_data.append(mae_improvements)
    
    improvement_array = np.array(improvement_data)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    sns.heatmap(improvement_array, 
                annot=True, fmt='.1f', 
                cmap='RdYlGn', center=0,
                xticklabels=horizons,
                yticklabels=['LightGBM', 'XGBoost', 'CatBoost', 'Ensemble'],
                cbar_kws={'label': 'Improvement over V4 Baseline (%)'},
                ax=ax)
    
    ax.set_title('MAE Improvement over Linear Regression V4 Baseline\n(Green = Better, Red = Worse)', fontsize=12)
    ax.set_xlabel('Forecast Horizon')
    ax.set_ylabel('Model')
    
    plt.tight_layout()
    plt.savefig('improvement_heatmap.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Saved improvement heatmap")
    
    cleanup()

print(f"Memory: {get_memory_usage():.2f} GB")

## 7. Bar Chart Comparison

In [ ]:
if results_summary:
    fig, axes = plt.subplots(2, 4, figsize=(16, 10))
    axes = axes.ravel()
    
    for idx, horizon in enumerate(horizons):
        if horizon in results_summary:
            ax = axes[idx]
            
            models = ['LGB', 'XGB', 'Cat', 'Ens', 'V4', 'V1']
            mae_values = [
                results_summary[horizon]['lgb']['mae'],
                results_summary[horizon]['xgb']['mae'],
                results_summary[horizon]['cat']['mae'],
                results_summary[horizon]['ensemble']['mae'],
                results_summary[horizon]['baseline_v4']['mae'],
                results_summary[horizon]['baseline_v1']['mae']
            ]
            
            colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd', '#d62728', '#e377c2']
            bars = ax.bar(models, mae_values, color=colors, alpha=0.7)
            
            v4_mae = results_summary[horizon]['baseline_v4']['mae']
            ax.axhline(y=v4_mae, color='red', linestyle='--', linewidth=1, alpha=0.5)
            
            for bar, val in zip(bars, mae_values):
                if val < v4_mae:
                    bar.set_edgecolor('green')
                    bar.set_linewidth(2)
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                       f'{val:.2f}', ha='center', va='bottom', fontsize=8)
            
            ax.set_title(f'{horizon} MAE', fontsize=10)
            ax.set_ylabel('MAE (μg/m³)')
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_ylim(0, max(mae_values) * 1.1)
    
    plt.suptitle('MAE Comparison Across All Models\n(Green border = Beats V4 Baseline)', fontsize=14)
    plt.tight_layout()
    plt.savefig('mae_bar_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Saved MAE bar comparison")
    
    cleanup()

print(f"Memory: {get_memory_usage():.2f} GB")

## 8. Single Sampled Scatter Plot

In [ ]:
def create_sampled_scatter(horizon='6h', sample_size=500):
    """Create a single sampled scatter plot for memory efficiency"""
    try:
        pred_file = f'../01_ensemble_training/trained/predictions_{horizon}.pkl'
        if not os.path.exists(pred_file):
            print(f"File not found: {pred_file}")
            return
        
        preds = joblib.load(pred_file)
        test_preds = preds['test']
        y_true = y_test[f'target_{horizon[:-1]}h']
        
        ensemble_pred = (test_preds['lgb'] + test_preds['xgb'] + test_preds['cat']) / 3
        
        sample_idx = np.random.RandomState(42).choice(len(y_true), 
                                                      min(sample_size, len(y_true)), 
                                                      replace=False)
        
        fig, ax = plt.subplots(figsize=(8, 8))
        
        ax.scatter(y_true[sample_idx], ensemble_pred[sample_idx], 
                  alpha=0.5, s=10, color='blue', label='Predictions')
        
        max_val = min(50, max(y_true[sample_idx].max(), ensemble_pred[sample_idx].max()))
        ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect prediction')
        
        mae = mean_absolute_error(y_true, ensemble_pred)
        r2 = r2_score(y_true, ensemble_pred)
        v4_mae = baseline_v4[horizon]['mae']
        v4_r2 = baseline_v4[horizon]['r2']
        
        ax.set_xlabel('True PM2.5 (μg/m³)', fontsize=12)
        ax.set_ylabel('Predicted PM2.5 (μg/m³)', fontsize=12)
        ax.set_title(f'Ensemble Predictions vs True Values - {horizon} Horizon\n' +
                    f'MAE: {mae:.3f} (vs V4: {v4_mae:.3f})\n' +
                    f'R²: {r2:.3f} (vs V4: {v4_r2:.3f})',
                    fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.set_xlim(0, max_val)
        ax.set_ylim(0, max_val)
        
        plt.tight_layout()
        plt.savefig(f'scatter_plot_{horizon}_sampled.png', dpi=100, bbox_inches='tight')
        plt.show()
        print(f"✓ Created sampled scatter plot for {horizon}")
        
        del preds, test_preds, ensemble_pred
        cleanup()
        
    except Exception as e:
        print(f"Error creating scatter plot: {e}")
        cleanup()

create_sampled_scatter('6h', sample_size=500)
print(f"Memory: {get_memory_usage():.2f} GB")

## 9. Statistical Significance Test

In [ ]:
from scipy import stats

def paired_t_test(horizon='6h'):
    """Perform paired t-test between ensemble and baseline"""
    try:
        pred_file = f'../01_ensemble_training/trained/predictions_{horizon}.pkl'
        if not os.path.exists(pred_file):
            return None
        
        preds = joblib.load(pred_file)
        test_preds = preds['test']
        y_true = y_test[f'target_{horizon[:-1]}h']
        
        ensemble_pred = (test_preds['lgb'] + test_preds['xgb'] + test_preds['cat']) / 3
        
        ensemble_errors = np.abs(y_true - ensemble_pred)
        
        sample_size = min(1000, len(ensemble_errors))
        sample_idx = np.random.RandomState(42).choice(len(ensemble_errors), sample_size, replace=False)
        
        sampled_errors = ensemble_errors[sample_idx]
        
        v4_mae = baseline_v4[horizon]['mae']
        ensemble_mae = np.mean(ensemble_errors)
        
        t_stat = (v4_mae - ensemble_mae) / (np.std(sampled_errors) / np.sqrt(sample_size))
        p_value = 2 * (1 - stats.t.cdf(abs(t_stat), sample_size - 1))
        
        del preds, test_preds, ensemble_pred, ensemble_errors
        gc.collect()
        
        return {
            'horizon': horizon,
            'ensemble_mae': ensemble_mae,
            'baseline_mae': v4_mae,
            'difference': v4_mae - ensemble_mae,
            't_statistic': t_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        }
        
    except Exception as e:
        print(f"Error in t-test: {e}")
        return None

print("Statistical Significance Tests (Ensemble vs V4 Baseline):")
print("="*60)
test_results = []

for horizon in ['1h', '6h', '12h', '24h']:
    result = paired_t_test(horizon)
    if result:
        test_results.append(result)
        sig = "✓ Significant" if result['significant'] else "✗ Not significant"
        print(f"{horizon}: Δ={result['difference']:.3f}, p={result['p_value']:.4f} {sig}")
    cleanup()

if test_results:
    test_df = pd.DataFrame(test_results)
    test_df.to_csv('statistical_tests.csv', index=False)
    print("\n✓ Saved statistical test results")

print(f"Memory: {get_memory_usage():.2f} GB")

## 10. Generate Summary Report

In [ ]:
print("="*70)
print("COMPREHENSIVE SUMMARY REPORT")
print("="*70)

if results_summary:
    print("\n1. ENSEMBLE PERFORMANCE vs BASELINES:")
    print("-"*50)
    
    beat_v4_mae = 0
    beat_v4_r2 = 0
    beat_v1_mae = 0
    beat_v1_r2 = 0
    
    for horizon in horizons:
        if horizon in results_summary:
            ens = results_summary[horizon]['ensemble']
            v4 = results_summary[horizon]['baseline_v4']
            v1 = results_summary[horizon]['baseline_v1']
            
            if ens['mae'] < v4['mae']:
                beat_v4_mae += 1
            if ens['r2'] > v4['r2']:
                beat_v4_r2 += 1
            if ens['mae'] < v1['mae']:
                beat_v1_mae += 1
            if ens['r2'] > v1['r2']:
                beat_v1_r2 += 1
    
    print(f"\nEnsemble vs V4 Baseline (485 hexagons, 2023-2025):")
    print(f"  - Beats V4 on MAE: {beat_v4_mae}/{len(horizons)} horizons")
    print(f"  - Beats V4 on R²:  {beat_v4_r2}/{len(horizons)} horizons")
    
    print(f"\nEnsemble vs V1 Baseline (Tokyo subset, 2023):")
    print(f"  - Beats V1 on MAE: {beat_v1_mae}/{len(horizons)} horizons")
    print(f"  - Beats V1 on R²:  {beat_v1_r2}/{len(horizons)} horizons")
    
    print("\n2. KEY FINDINGS:")
    print("-"*50)
    
    short_term = ['1h', '2h', '3h']
    long_term = ['12h', '24h']
    
    short_improvement = np.mean([((results_summary[h]['baseline_v4']['mae'] - 
                                   results_summary[h]['ensemble']['mae']) / 
                                  results_summary[h]['baseline_v4']['mae'] * 100) 
                                 for h in short_term if h in results_summary])
    
    long_improvement = np.mean([((results_summary[h]['baseline_v4']['mae'] - 
                                  results_summary[h]['ensemble']['mae']) / 
                                 results_summary[h]['baseline_v4']['mae'] * 100) 
                                for h in long_term if h in results_summary])
    
    print(f"  • Short-term (1-3h) MAE improvement: {short_improvement:+.1f}%")
    print(f"  • Long-term (12-24h) MAE improvement: {long_improvement:+.1f}%")
    
    best_model_counts = {'lgb': 0, 'xgb': 0, 'cat': 0, 'ensemble': 0}
    for horizon in results_summary:
        best_mae = float('inf')
        best_model = ''
        for model in best_model_counts.keys():
            if results_summary[horizon][model]['mae'] < best_mae:
                best_mae = results_summary[horizon][model]['mae']
                best_model = model
        best_model_counts[best_model] += 1
    
    print(f"\n  • Best performing model by horizon:")
    for model, count in best_model_counts.items():
        if count > 0:
            print(f"    - {model.upper()}: {count} horizons")
    
    print("\n3. RECOMMENDATIONS:")
    print("-"*50)
    if short_improvement > 0:
        print("  ✓ Ensemble shows improvement for short-term predictions")
    else:
        print("  ⚠ Consider tuning ensemble weights for short-term predictions")
    
    if long_improvement > 0:
        print("  ✓ Ensemble shows improvement for long-term predictions")
    else:
        print("  ⚠ Long-term predictions need architectural improvements")
    
    if beat_v4_r2 < len(horizons) / 2:
        print("  ⚠ R² scores indicate potential overfitting - consider regularization")

    summary_json = {
        'results': results_summary,
        'summary': {
            'beat_v4_mae': beat_v4_mae,
            'beat_v4_r2': beat_v4_r2,
            'beat_v1_mae': beat_v1_mae,
            'beat_v1_r2': beat_v1_r2,
            'short_term_improvement': short_improvement,
            'long_term_improvement': long_improvement
        }
    }
    
    with open('memory_optimized_summary.json', 'w') as f:
        json.dump(summary_json, f, indent=2)
    
    print("\n✓ Summary saved to memory_optimized_summary.json")

print(f"\nFinal memory usage: {get_memory_usage():.2f} GB")
print("✓ Analysis complete!")

## 11. Cleanup and Memory Report

In [ ]:
print("Performing final cleanup...")
print(f"Memory before cleanup: {get_memory_usage():.2f} GB")

del results_summary, y_test
if 'mae_df' in locals():
    del mae_df
if 'r2_df' in locals():
    del r2_df
if 'test_df' in locals():
    del test_df

plt.close('all')
gc.collect()

print(f"Memory after cleanup: {get_memory_usage():.2f} GB")
print("\nGenerated files:")
print("  - mae_comparison_optimized.csv")
print("  - r2_comparison_optimized.csv")
print("  - performance_trends_lightweight.png")
print("  - improvement_heatmap.png")
print("  - mae_bar_comparison.png")
print("  - scatter_plot_6h_sampled.png")
print("  - statistical_tests.csv")
print("  - memory_optimized_summary.json")
print("\n✓ Notebook execution complete!")